<a href="https://colab.research.google.com/github/Anmolsharma44/AI-LLM-IITJammu/blob/main/imagegeneration/iMAGEhUGGING.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install diffusers, transformers, accelerate, and safetensors for fast model loading
!pip install --quiet diffusers transformers accelerate safetensors torch

In [ ]:
import torch
from diffusers import DiffusionPipeline

# Define the model ID (Stable Diffusion XL v1.0 Base)
model_id = "stabilityai/stable-diffusion-xl-base-1.0"

print(f"Loading {model_id} onto GPU... this may take a moment.")

# 1. Load the pipeline with FP16 precision (essential to fit on T4 GPU)
# Use variant="fp16" to ensure the model weights are optimized for GPU memory
pipe = DiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    use_safetensors=True,
    variant="fp16"
)

# 2. Move the pipeline to the GPU (CUDA device)
pipe.to("cuda")

# Optional: Optimize for lower memory usage if you hit VRAM limits
# pipe.enable_model_cpu_offload() # Enable this if you get CUDA OOM errors

print("Pipeline initialized and ready.")

Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Loading stabilityai/stable-diffusion-xl-base-1.0 onto GPU... this may take a moment.


model_index.json:   0%|          | 0.00/609 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 19 files:   0%|          | 0/19 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

AssertionError: Torch not compiled with CUDA enabled

In [ ]:
# Define your detailed text prompt
prompt = "A cinematic, hyperrealistic photograph of a futuristic cyberpunk cityscape at night, rain-slicked neon streets reflecting holographic advertisements, towering skyscrapers, bustling crowds, atmosphere, 8k resolution, detailed."

# Run inference (returns a PIL image object)
print(f"Generating image for prompt: '{prompt}'")
image = pipe(prompt).images[0]

# Save the generated image locally in Colab
image.save("generated_image.png")
print("Image saved as 'generated_image.png'")

In [ ]:
from IPython.display import display

# Display the PIL image object directly in the notebook output
display(image)

In [ ]:
from diffusers import DiffusionPipeline
import torch

# Load Base Model (as before)
model_id = "stabilityai/stable-diffusion-xl-base-1.0"
print("Loading SDXL Base...")
pipe = DiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    use_safetensors=True,
    variant="fp16",
)
pipe.to("cuda")

# Load Refiner Model (specify the same variant)
refiner_id = "stabilityai/stable-diffusion-xl-refiner-1.0"
print("Loading SDXL Refiner...")
refiner = DiffusionPipeline.from_pretrained(
    refiner_id,
    torch_dtype=torch.float16,
    use_safetensors=True,
    variant="fp16",
)
refiner.to("cuda")

# Define Prompt
prompt = "An intricate macro photograph of a glowing bioluminescent mushroom in a dark, mossy ancient forest, ethereal lighting, depth of field, photorealistic 8k."

# --- Run the Two-Stage Pipeline ---

# Step 1: Base model generates latent images (we control the denoising strength)
print("Running Base Model...")
latents = pipe(
    prompt,
    output_type="latent", # Stop early to pass to refiner
    num_inference_steps=40 # Good balance of speed/quality
).images

# Step 2: Refiner model processes the latents to add fine details
print("Running Refiner Model...")
refined_image = refiner(
    prompt=prompt,
    image=latents, # Pass the latents from the base
    denoising_start=0.8 # Define where the refiner takes over (usually 0.8)
).images[0]

# Save and Display
refined_image.save("refined_image.png")
display(refined_image)